In [22]:
import zipfile
import rasterio
import os
from rasterio.enums import Resampling
import numpy as np
import matplotlib.pyplot as plt
import glob
import warnings
import pandas as pd
import re






In [2]:
RAW_DATA_PATH       = r"F:\data_water\raw"
PROCESSED_DATA_PATH = r"F:\data_water\processed"
RAW_EXTRACTED       = r"F:\data_water\raw_extracted"

In [3]:
for scenes in os.listdir(RAW_DATA_PATH):
  if scenes.endswith(".zip"):
    if os.path.exists(os.path.join(RAW_EXTRACTED, scenes.replace('.zip', ''))):
      print(scenes, "already extracted")
      continue

    with zipfile.ZipFile((os.path.join(RAW_DATA_PATH, scenes)), "r") as zip_ref:
      zip_ref.extractall(RAW_EXTRACTED)
    print(f"File {scenes} extracted")

    # break     # just to check for 1 file

S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T134053.SAFE.zip already extracted
S2A_MSIL2A_20240330T090551_N0510_R050_T34SGE_20240330T121454.SAFE.zip already extracted
S2A_MSIL2A_20240127T090231_N0510_R007_T34SGE_20240127T121650.SAFE.zip already extracted
S2A_MSIL2A_20240206T090141_N0510_R007_T34SGE_20240206T121453.SAFE.zip already extracted
S2A_MSIL2A_20240117T090311_N0510_R007_T34SGE_20240117T121052.SAFE.zip already extracted
S2B_MSIL2A_20240322T085629_N0510_R007_T34SGE_20240322T114528.SAFE.zip already extracted
File S2A_MSIL2A_20240310T090801_N0510_R050_T34SGE_20240310T122650.SAFE.zip extracted
File S2B_MSIL2A_20240224T090859_N0510_R050_T34SGE_20240224T104230.SAFE.zip extracted
File S2A_MSIL2A_20240327T085601_N0510_R007_T34SGE_20240327T122850.SAFE.zip extracted
File S2B_MSIL2A_20240630T085559_N0510_R007_T34SGE_20240630T105345.SAFE.zip extracted
File S2A_MSIL2A_20240608T090601_N0510_R050_T34SGE_20240608T122944.SAFE.zip extracted
File S2B_MSIL2A_20240404T090549_N0510_R050_T34S

In [ ]:
print("NDWI min/max:", np.nanmin(band1), np.nanmax(band1))
print("Turbidity min/max:", np.nanmin(turbidity), np.nanmax(turbidity))
print("Chlorophyll min/max:", np.nanmin(chlorophyll), np.nanmax(chlorophyll))
print("ratio min/max:", np.nanmin(ratio), np.nanmax(ratio))
print("b02/b03 ratio min/max:", np.nanmin(b02_final/b03_final), np.nanmax(b02_final/b03_final))

In [ ]:
indices_path = PROCESSED_DATA_PATH / "indices"
os.makedirs(indices_path, exist_ok=True)
bands_path = PROCESSED_DATA_PATH / "bands"
os.makedirs(bands_path, exist_ok=True)
rgb_path = PROCESSED_DATA_PATH / "rgb"
os.makedirs(rgb_path, exist_ok=True)

In [ ]:
for scenes in os.listdir(RAW_EXTRACTED):
  if scenes.endswith('.SAFE'):
    date = scenes[11:19]
    name_indices = f"Indices_{date}_processed.tif"
    name_bands = f"Bands_{date}_processed.tif"
    name_rgb = f"Bands_{date}_processed.png"
    out_indices = indices_path / name_indices
    out_bands =bands_path /  name_bands
    out_rgb = rgb_path /  name_rgb


    if os.path.exists(out_indices) and os.path.exists(out_bands) and os.path.exists(out_rgb):
      print(f"File {scenes} already processed")
      continue
 
    else:
      print(f"Starting processing {date} .. .tif")

      search_b04 = glob.glob(os.path.join(RAW_EXTRACTED, scenes, '**/*_B04_10m.jp2'), recursive = True) # 04 - Red, Used for Turbidity
      search_b02 = glob.glob(os.path.join(RAW_EXTRACTED, scenes, '**/*_B02_10m.jp2'), recursive = True) # 02 - Blue, Used for Chlorophyll and Bathymetry
      search_b03 = glob.glob(os.path.join(RAW_EXTRACTED, scenes, '**/*_B03_10m.jp2'), recursive = True) # 03 - Green, Used for Chlorophyll and Bathymetry
      search_b08 = glob.glob(os.path.join(RAW_EXTRACTED, scenes, '**/*_B08_10m.jp2'), recursive = True) # 08 - NIR, Glint removal
      search_scl = glob.glob(os.path.join(RAW_EXTRACTED, scenes, '**/*_SCL_20m.jp2'), recursive=True)   # SCL for water masks(scl == 6)

      print("** Starting scl **")
      with rasterio.open(search_scl[0]) as src:
        scl = src.read(1, out_shape=(10980, 10980), resampling=Resampling.nearest).astype('float32')
        profile = src.profile
        print("SCL shape:", scl.shape)
        # print(profile)
        print("** scl done **")
      valid_masks = (scl == 6) # 6 is water , so keeps only water pixels
      

      print("Starting b08")
      with rasterio.open(search_b08[0]) as src:
        b08 = src.read(1).astype('float32')

        ## REFLECTANCE (default unit) is: DN = 10000 * REFLECTANCE, for physical algorithms (chlorophyll, turbidity, bathymetry) convert to reflectance first, Coefficients assume 0-1 range

        b08_reflectance = b08 / 10000
        b08_masked = np.where(valid_masks, b08_reflectance, np.nan)
        print("** b08 done **\n")

      print("** Starting b02 **")

      glint_factor = 0.5

      with rasterio.open(search_b02[0]) as src:
        b02 = src.read(1).astype('float32')
        profile = src.profile
        print("b02 shape: ", b02.shape)
        # print(profile)
        print("b02 min:", np.nanmin(b02))
        print("b02 max:", np.nanmax(b02))

        b02_reflectance = b02 / 10000
        b02_masked = np.where(valid_masks, b02_reflectance, np.nan)

        # NIR over deep water should be ~0
        # Any NIR signal = glint
        # Subtract from visible bands # a glint factor of 0.5 is used after trying subtracting nir since a lot of pixels was going negative and then nan
        b02_corrected = b02_masked - (glint_factor * b08_masked)    
        b02_corrected = np.where(b02_corrected < 0, np.nan, b02_corrected)  
        print("b02_corrected min:", np.nanmin(b02_corrected))
        print("b02_corrected max:", np.nanmax(b02_corrected))

        ##Total valid pixels
        print("Total valid pixels for b02: ", np.sum(~np.isnan(b02_corrected)))
        print("Total pixels ob b02: ", b02_corrected.size)
        print("water percentage:", round(np.sum(~np.isnan(b02_corrected)) / b02_corrected.size * 100, 2), "%")

        print("** b02 done **\n")

      print("** Starting b03 **")
      with rasterio.open(search_b03[0]) as src:
        b03 = src.read(1).astype('float32')
        profile = src.profile
        print("b03 shape: ",b03.shape)
        # print(profile)
        print("b03 min:", b03.min())
        print("b03 max:", b03.max())

        b03_reflectance = b03 / 10000
        b03_masked = np.where(valid_masks, b03_reflectance, np.nan)

        #glint removal
        b03_corrected = b03_masked - (glint_factor * b08_masked  )   
        b03_corrected = np.where(b03_corrected < 0, np.nan, b03_corrected)  
        print("b03_corrected min:", np.nanmin(b03_corrected))
        print("b03_corrected max:", np.nanmax(b03_corrected))

         ##Total valid pixels
        print("Total valid pixels for b03: ", np.sum(~np.isnan(b03_corrected)))
        print("Total pixels ob b03: ", b03_corrected.size)
        print("water percentage:", round(np.sum(~np.isnan(b03_corrected)) / b03_corrected.size * 100, 2), "%")


        print("** b03 done **\n")

      
      print("** Starting b04 **")
      with rasterio.open(search_b04[0]) as src:
        b04 = src.read(1).astype('float32')
        out_profile = src.profile        # profile of b04 is chosen since its one of 10 m resolution and the out meta data should be 10m res
        print("b04 shape:",b04.shape)
        # print(profile)
        print("b04 min:", b04.min())
        print("b04 max:", b04.max())

        b04_reflectance = b04 / 10000
        b04_masked = np.where(valid_masks, b04_reflectance, np.nan)

        #glint removal
        b04_corrected = b04_masked - (glint_factor * b08_masked )    
        b04_corrected = np.where(b04_corrected < 0, np.nan, b04_corrected)  
        print("b04_corrected min:", np.nanmin(b04_corrected))
        print("b04_corrected max:", np.nanmax(b04_corrected))

        ##Total valid pixels
        print("Total valid pixels for b04: ", np.sum(~np.isnan(b04_corrected)))
        print("Total pixels ob b04: ", b04_corrected.size)
        print("water percentage:", round(np.sum(~np.isnan(b04_corrected)) / b04_corrected.size * 100, 2), "%")

        print("** b04 done **\n")

      combined_mask = (
        ~np.isnan(b02_corrected) &
        ~np.isnan(b03_corrected) &
        ~np.isnan(b04_corrected)
      )

      b08_final = np.where(combined_mask, b08_masked, np.nan)
      b02_final = np.where(combined_mask, b02_corrected, np.nan)
      b03_final = np.where(combined_mask, b03_corrected, np.nan)
      b04_final = np.where(combined_mask, b04_corrected, np.nan)

      print("Final consistent pixels: ", combined_mask.sum())
      # print("Percentage:", round(combined_mask.sum() / b02_corrected.size * 100, 2), "%")

      # INDICES Calculation for LSTM (NDWI, Chlorophyll and turbidity)

      #Normalise water index
      NDWI = (b03_final - b08_final) / (b03_final + b08_final)

      # Based on Dogliotti et al. 2015, Turbidity
      turbidity = 1.73 * b04_final / (1 - b04_final / 0.17)

      # Based on OC2 algorithm (O'Reilly et al.)
      ratio = np.log10(b02_final / b03_final)
      chlorophyll = 10 ** (0.2424 - 2.7423 * ratio + 1.8017 * ratio**2)

      out_profile['dtype'] = 'float32'
      out_profile['count'] = 3
      out_profile['nodata'] = np.nan
      out_profile['driver'] = 'GTiff'

      with rasterio.open(out_indices, 'w', **out_profile) as dst:
        dst.write(NDWI, 1)
        dst.write(turbidity, 2)
        dst.write(chlorophyll, 3)
      print(f"Indices for {date} Completed")

      with rasterio.open(out_bands, 'w', **out_profile) as dst:
        dst.write(b02_final, 1)
        dst.write(b03_final, 2)
        dst.write(b04_final, 3)
      print(f"Bands for {date} Completed")

      rgb = np.dstack([b04_reflectance, b03_reflectance, b02_reflectance])
      # Percentile stretch for better contrast
      for i in range(3):
          p2  = np.nanpercentile(rgb[:,:,i], 2)
          p98 = np.nanpercentile(rgb[:,:,i], 98)
          rgb[:,:,i] = np.clip((rgb[:,:,i] - p2) / (p98 - p2), 0, 1)
      
      plt.figure(figsize=(10, 10))
      plt.imshow(rgb)
      plt.axis('off')
      plt.title(f'True Color RGB')
      plt.tight_layout()
      plt.savefig(out_rgb, dpi=150, bbox_inches='tight')
      plt.close()

    break

print("ALL FILES PROCESSING COMPLETED")






Starting processing 20240330 .. .tif
** Starting scl **
SCL shape: (10980, 10980)
** scl done **
Starting b08
** b08 done **

** Starting b02 **
b02 shape:  (10980, 10980)
b02 min: 0.0
b02 max: 21472.0
b02_corrected min: 0.0
b02_corrected max: 0.43595004
Total valid pixels for b02:  95225830
Total pixels ob b02:  120560400
water percentage: 78.99 %
** b02 done **

** Starting b03 **
b03 shape:  (10980, 10980)
b03 min: 0.0
b03 max: 19808.0
b03_corrected min: 0.0
b03_corrected max: 0.4902
Total valid pixels for b03:  95230058
Total pixels ob b03:  120560400
water percentage: 78.99 %
** b03 done **

** Starting b04 **
b04 shape: (10980, 10980)
b04 min: 0.0
b04 max: 18736.0
b04_corrected min: 0.0
b04_corrected max: 0.66200006
Total valid pixels for b04:  95230662
Total pixels ob b04:  120560400
water percentage: 78.99 %
** b04 done **

Final consistent pixels:  95224473


C:\Users\haris\AppData\Local\Temp\ipykernel_68528\2963705295.py:147: RuntimeWarning: divide by zero encountered in divide
  turbidity = 1.73 * b04_final / (1 - b04_final / 0.17)
C:\Users\haris\AppData\Local\Temp\ipykernel_68528\2963705295.py:150: RuntimeWarning: divide by zero encountered in log10
  ratio = np.log10(b02_final / b03_final)


Indices for 20240330 Completed
Bands for 20240330 Completed
ALL FILES PROCESSING COMPLETED


In [17]:

bands_folder = os.path.join(PROCESSED_DATA_PATH, "bands")

band_files = os.listdir(bands_folder)

band_path = os.path.join(bands_folder, band_files[0])

# Check one bands file
with rasterio.open(band_path) as src:
    data = src.read()

print("=== BANDS CHECK ===")
print(f"Shape: {data.shape}")
print(f"B02 (Blue)  min/max: {np.nanmin(data[0]):.4f} / {np.nanmax(data[0]):.4f}")
print(f"B03 (Green) min/max: {np.nanmin(data[1]):.4f} / {np.nanmax(data[1]):.4f}")
print(f"B04 (Red)   min/max: {np.nanmin(data[2]):.4f} / {np.nanmax(data[2]):.4f}")

nan_b02 = np.isnan(data[0]).sum()
nan_b03 = np.isnan(data[1]).sum()
nan_b04 = np.isnan(data[2]).sum()

print(f"NaN B02: {nan_b02}")
print(f"NaN B03: {nan_b03}")
print(f"NaN B04: {nan_b04}")
print(f"Consistent: {nan_b02 == nan_b03 == nan_b04}")

=== BANDS CHECK ===
Shape: (3, 10980, 10980)
B02 (Blue)  min/max: 0.0000 / 0.4360
B03 (Green) min/max: 0.0005 / 0.4902
B04 (Red)   min/max: 0.0083 / 0.6620
NaN B02: 25335927
NaN B03: 25335927
NaN B04: 25335927
Consistent: True


In [18]:
indices_folder = os.path.join(PROCESSED_DATA_PATH, "indices")

indices_files = os.listdir(indices_folder)

indices_path = os.path.join(indices_folder, indices_files[0])

with rasterio.open(indices_path) as src:
    idx = src.read()

print("=== INDICES CHECK ===")
print(f"NDWI    min/max: {np.nanmin(idx[0]):.4f} / {np.nanmax(idx[0]):.4f}")
print(f"Turbidity min/max: {np.nanmin(idx[1]):.4f} / {np.nanmax(idx[1]):.4f}")
print(f"Chlorophyll min/max: {np.nanmin(idx[2]):.4f} / {np.nanmax(idx[2]):.4f}")

=== INDICES CHECK ===
NDWI    min/max: -0.9975 / 1.0000
Turbidity min/max: 0.0151 / 99.7014
Chlorophyll min/max: 0.2988 / 7.2925


In [24]:
indices_folder = os.path.join(PROCESSED_DATA_PATH, "indices")

# List only tif files
indices_files = [
    f for f in os.listdir(indices_folder)
    if f.endswith(".tif")
]

records = []

for f in sorted(indices_files):

    # Extract date from filename
    match = re.search(r"\d{8}", f)

    if match is None:
        print("Skipping file, no date found:", f)
        continue

    date = match.group()

    # Full path to this tif file
    file_path = os.path.join(indices_folder, f)

    with rasterio.open(file_path) as src:
        idx = src.read().astype(float)

        if src.nodata is not None:
            idx[idx == src.nodata] = np.nan

    records.append({
        "date": date,
        "ndwi_mean": np.nanmean(idx[0]),
        "turb_mean": np.nanmean(idx[1]),
        "chl_mean": np.nanmean(idx[2]),
        "valid_pct": round((~np.isnan(idx[0])).sum() / idx[0].size * 100, 2)
    })

df = pd.DataFrame(records)

df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df = df.sort_values("date")

print(df.to_string(index=False))

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# NDWI
axes[0].plot(df["date"], df["ndwi_mean"], marker="o", color="blue", linewidth=1.5)
axes[0].set_ylabel("NDWI Mean")
axes[0].set_title("Water Quality Time Series — Crete South Coast")
axes[0].grid(True, alpha=0.3)
axes[0].axhline(df["ndwi_mean"].mean(), color="blue", linestyle="--", alpha=0.5, label="mean")
axes[0].legend()

# Turbidity
axes[1].plot(df["date"], df["turb_mean"], marker="o", color="orange", linewidth=1.5)
axes[1].set_ylabel("Turbidity Mean")
axes[1].grid(True, alpha=0.3)
axes[1].axhline(df["turb_mean"].mean(), color="orange", linestyle="--", alpha=0.5, label="mean")
axes[1].legend()

# Chlorophyll
axes[2].plot(df["date"], df["chl_mean"], marker="o", color="green", linewidth=1.5)
axes[2].set_ylabel("Chlorophyll Mean")
axes[2].set_xlabel("Date")
axes[2].grid(True, alpha=0.3)
axes[2].axhline(df["chl_mean"].mean(), color="green", linestyle="--", alpha=0.5, label="mean")
axes[2].legend()

plt.tight_layout()
plt.savefig("time_series_check.png", dpi=150, bbox_inches="tight")
plt.show()

RasterioIOError: F:\data_water\processed\indices\indices 20250910.tif: No such file or directory